In [ ]:
import os.path as osp

import numpy as np
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from wofostat.wofost import WOFOST, DEFAULT_PARAMETER_VALUES
from wofostat import end_of_season_sensitivity_func, run_sensitivity_analysis, get_parameter_spec
from pcse.base import ParameterProvider

In [ ]:
netherlands_wdp = NASAPowerWeatherDataProvider(latitude=51, longitude=5)
print(netherlands_wdp)

In [ ]:
india_wdp = NASAPowerWeatherDataProvider(latitude=23, longitude=73)
print(india_wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
netherlands_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_netherlands_2021.agro"))
print(netherlands_agro)

In [ ]:
india_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_india_2021.agro"))
print(india_agro)

In [ ]:
params = WOFOST.get_params(cropd=cropd, sited=sited, soild=soild)
wofost = WOFOST(params, netherlands_wdp, netherlands_agro)

params = WOFOST.override(DEFAULT_PARAMETER_VALUES, params)
observed_data = wofost.run()
observed_data

In [ ]:
observed_data.plot.scatter(x="day", y="LAI")

In [ ]:
observed_data.plot.scatter(x="day", y="DVS")

In [ ]:
observed_data.plot.scatter(x="day", y="TWSO")

In [ ]:
data_table = [
    {
        "name": "TSUM1",
        "distribution": "uniform",
        "range": "150, 280",
        "type": "continuous",
    },
    {
        "name": "TSUM2",
        "distribution": "uniform",
        "range": "1550, 2100",
        "type": "continuous",
    },
    {
        "name": "TBASEM",
        "distribution": "uniform",
        "range": "2, 4",
        "type": "continuous",
    },
    {
        "name": "TSUMEM",
        "distribution": "uniform",
        "range": "170, 255",
        "type": "continuous",
    },
    {
        "name": "TEFFMX",
        "distribution": "uniform",
        "range": "18, 32",
        "type": "continuous",
    },
    {
        "name": "SPAN",
        "distribution": "uniform",
        "range": "20, 50",
        "type": "continuous",
    },
    {
        "name": "TDWI",
        "distribution": "uniform",
        "range": "75, 700",
        "type": "continuous",
    },
    {
        "name": "RGRLAI",
        "distribution": "uniform",
        "range": "0.008, 0.02",
        "type": "continuous",
    },
    {
        "name": "Q10",
        "distribution": "uniform",
        "range": "2, 3",
        "type": "continuous",
    },
] 

parameter_spec = get_parameter_spec(data_table)
parameter_spec

In [ ]:
state_vars = ["LAI", "TWSO"]

run_sensitivity_analysis(
    experiment_name = "netherlands_sensitivity_analysis",
    parameter_spec=parameter_spec,
    n_samples = 256,
    wdp = netherlands_wdp,
    agro=netherlands_agro,
    state_vars = state_vars,
    calibration_func = end_of_season_sensitivity_func,
    params = params,
    n_jobs = 10
)

In [ ]:
run_sensitivity_analysis(
    experiment_name = "india_sensitivity_analysis",
    parameter_spec=parameter_spec,
    n_samples = 256,
    wdp = india_wdp,
    agro=india_agro,
    state_vars = state_vars,
    calibration_func = end_of_season_sensitivity_func,
    params = params,
    n_jobs = 10
)